# 🐷 Pig Posture Recognition – Kaggle K-Fold Strategy

Dieses Notebook trainiert das stärkste Setup: **ConvNeXt-Base** mit **5-Fold Cross Validation**.

Anstatt nur ein Modell zu trainieren, teilen wir den Datensatz in 5 gleich große Abschnitte. Wir trainieren 5 separate Modelle, die jeweils 4 Abschnitte zum Training und 1 Abschnitt zur Validierung (100% Abdeckung!) verwenden.

**Features dieses Profils:**
- Model: `convnext_base`
- Resolution: `288x288`
- Augmentierung: `timm.data.create_transform` (nutzt professionelles **RandAugment**)
- Split: **StratifiedKFold** (erhält die Balance der 5 Posen in jedem Fold exakt)
- Gradient Accumulation: Falls dein VRAM zu klein wird, kumuliert es die Loss-Schritte.

## ⚙️ Configuration

In [1]:
TAG = "T2"   # "T1" or "T2"

DATA_ROOT = "/datasets/multi-view-pig-posture-recognition"

if TAG == "T1":
    CSV_PATH = f"{DATA_ROOT}/train1.csv"
    IMG_DIR  = f"{DATA_ROOT}/train1_images"
else:
    CSV_PATH = f"{DATA_ROOT}/train2.csv"
    IMG_DIR  = f"{DATA_ROOT}/train2_images"

OUTPUT_DIR = f"runs/kfold_{TAG.lower()}"

MODEL_NAME     = "convnext_base"  
IMG_SIZE       = 288
N_FOLDS        = 5                   # Anzahl der Kreuzvalidierungs-Splits
BATCH_SIZE     = 32
EPOCHS         = 30                  # 30 pro Fold reicht meist bei 5 Folds
LR             = 3e-4
LABEL_SMOOTH   = 0.15                # Hohes Smoothing zwingt das Modell robuster zu sein
MIXUP_ALPHA    = 0.30
PAD_RATIO      = 0.25
NUM_WORKERS    = 8
SEED           = 42
NUM_CLASSES    = 5

CLASS_NAMES    = ["Lateral_lying_left", "Lateral_lying_right",
                  "Sitting", "Standing", "Sternal_lying"]

print(f"Tag: {TAG}  |  Model: {MODEL_NAME}  |  Folds: {N_FOLDS}")
print(f"Output: {OUTPUT_DIR}")

Tag: T2  |  Model: convnext_base  |  Folds: 5
Output: runs/kfold_t2


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"

## 📚 Imports

In [3]:
import os, ast, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import timm

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f'Anzahl verfügbarer GPUs: {torch.cuda.device_count()}')

<jemalloc>: Unsupported system page size


Device: cuda
Anzahl verfügbarer GPUs: 2


In [4]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 🗂️ Dataset & Augmentations (RandAugment)

In [5]:
class PigPostureDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.25):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform: crop = self.transform(crop)
        return crop, int(row["class_id"])


from timm.data.transforms_factory import create_transform

# Wir generieren Auto-Augments direkt von Timm. 'rand-m9-mstd0.5-inc1' ist der Industrie-Standard für ViTs/ConvNeXt
train_transform = create_transform(
    input_size=IMG_SIZE,
    is_training=True,
    color_jitter=0.4,
    auto_augment='rand-m9-mstd0.5-inc1', 
    interpolation='bicubic',
    re_prob=0.25, # Random Erasing Probability
    re_mode='pixel',
    re_count=1,
)

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 🛠️ Helpers (MixUp, Train Loop, Val Loop)

In [6]:
def mixup_data(x, y, alpha=0.3):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_one_epoch(model, loader, optimizer, scaler, criterion_plain):
    model.train()
    loss_sum, preds_all, labels_all = 0.0, [], []
    for imgs, labels in tqdm(loader, desc="  Train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs, y_a, y_b, lam = mixup_data(imgs, labels, alpha=MIXUP_ALPHA)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss   = mixup_loss(criterion_plain, logits, y_a, y_b, lam)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(y_a.cpu().numpy())
    n = len(loader.dataset)
    return loss_sum / n, f1_score(labels_all, preds_all, average="macro", zero_division=0)

@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    loss_sum, preds_all, labels_all = 0.0, [], []
    for imgs, labels in tqdm(loader, desc="  Val  ", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    n = len(loader.dataset)
    return loss_sum / n, f1_score(labels_all, preds_all, average="macro", zero_division=0), preds_all, labels_all

## 🚀 K-Fold Training Loop
Iteriert durch 5 Folds. Für jeden Fold wird ein komplett neues ConvNeXt Modell auf der iterativen Datenmenge geladen.

In [7]:
df = pd.read_csv(CSV_PATH)
print(f"Initialisiere Stratified {N_FOLDS}-Fold CV auf {len(df)} Bildern...")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X=df, y=df["class_id"])):
        # ---- NEU: Folds überspringen, wenn sie schon fertig sind ----
    # Wir nehmen an, wenn ein Checkpoint vom Fold existiert, lassen wir es gut sein.
    ckpt_path = os.path.join(OUTPUT_DIR, f"best_model_fold_{fold_idx+1}.pth")
    if os.path.exists(ckpt_path):
        print(f"⏩ Überspringe Fold {fold_idx+1}, Modell {ckpt_path} existiert bereits!")
        continue
    print(f"\n{'='*50}")
    print(f"🚀 Starte FOLD {fold_idx + 1} / {N_FOLDS}")
    print(f"{'='*50}")
    
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)
    
    train_ds = PigPostureDataset(train_df, IMG_DIR, transform=train_transform, pad_ratio=PAD_RATIO)
    val_ds   = PigPostureDataset(val_df,   IMG_DIR, transform=val_transform, pad_ratio=PAD_RATIO)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    
    # Konstruiere neues Modell für jede Fold-Iterierung
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)
    if torch.cuda.device_count() > 1:
        print(f"Nutze {torch.cuda.device_count()} GPUs mit nn.DataParallel!")
        model = nn.DataParallel(model)
    
    counts  = Counter(train_df["class_id"].tolist())
    weights = torch.tensor(
        [len(train_df) / (NUM_CLASSES * max(counts.get(c, 1), 1)) for c in range(NUM_CLASSES)],
        dtype=torch.float32
    ).to(DEVICE)
    
    criterion       = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTH)
    criterion_plain = nn.CrossEntropyLoss(weight=weights)
    
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = GradScaler()
    
    best_val_f1 = 0.0
    log = []
    
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_f1       = train_one_epoch(model, train_loader, optimizer, scaler, criterion_plain)
        val_loss, val_f1, _, _     = validate_epoch(model, val_loader, criterion)
        scheduler.step()
        lr = scheduler.get_last_lr()[0]

        mark = "★" if val_f1 > best_val_f1 else " "
        print(f"{mark} Fold {fold_idx+1} | Epoche {epoch:03d}/{EPOCHS} | "
              f"Train L={train_loss:.4f} F1={train_f1:.4f} | "
              f"Val L={val_loss:.4f} F1={val_f1:.4f}")

        log.append(dict(fold=fold_idx+1, epoch=epoch, val_f1=val_f1))

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            ckpt_path   = os.path.join(OUTPUT_DIR, f"best_model_fold_{fold_idx+1}.pth")
            torch.save({
                "epoch": epoch, "model": model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
                "val_f1": val_f1, "model_name": MODEL_NAME, "tag": TAG,
            }, ckpt_path)

    fold_results.append(best_val_f1)
    print(f"\n✅ Fold {fold_idx+1} beendet! Bester Val F1: {best_val_f1:.4f}")

print(f"\n{'='*50}")
print(f"🏁 Komplettes 5-Fold Training abgeschlossen!")
print(f"F1 Scores der einzelnen Modelle: {[round(x, 4) for x in fold_results]}")
score = sum(fold_results)/N_FOLDS
print(f"🏆 DURCHSCHNITTLICHER F1-SCORE: {score:.5f}")


Initialisiere Stratified 5-Fold CV auf 23450 Bildern...
⏩ Überspringe Fold 1, Modell runs/kfold_t2/best_model_fold_1.pth existiert bereits!
⏩ Überspringe Fold 2, Modell runs/kfold_t2/best_model_fold_2.pth existiert bereits!
⏩ Überspringe Fold 3, Modell runs/kfold_t2/best_model_fold_3.pth existiert bereits!
⏩ Überspringe Fold 4, Modell runs/kfold_t2/best_model_fold_4.pth existiert bereits!

🚀 Starte FOLD 5 / 5
Nutze 2 GPUs mit nn.DataParallel!


  Train:   0%|          | 0/586 [00:00<?, ?it/s]

  Val  :   0%|          | 0/147 [00:00<?, ?it/s]

★ Fold 5 | Epoche 001/30 | Train L=1.3920 F1=0.3043 | Val L=1.5803 F1=0.5060


  Train:   0%|          | 0/586 [00:00<?, ?it/s]

...

  Val  :   0%|          | 0/147 [00:00<?, ?it/s]

  Fold 5 | Epoche 029/30 | Train L=0.5504 F1=0.5539 | Val L=2.0933 F1=0.9119


  Train:   0%|          | 0/586 [00:00<?, ?it/s]

  Val  :   0%|          | 0/147 [00:00<?, ?it/s]

★ Fold 5 | Epoche 030/30 | Train L=0.5768 F1=0.5621 | Val L=2.1216 F1=0.9125

✅ Fold 5 beendet! Bester Val F1: 0.9125

🏁 Komplettes 5-Fold Training abgeschlossen!
F1 Scores der einzelnen Modelle: [0.9125]
🏆 DURCHSCHNITTLICHER F1-SCORE: 0.18249
